In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
import ast
import random
import warnings
from datetime import datetime
from collections import Counter
import joblib  # For saving the scaler
import psutil

# Suppress warnings
warnings.filterwarnings('ignore')

class Config:
    # Data configuration
    features_dim = 256
    num_classes = 2
    
    # Model architecture
    num_stages = 2
    num_layers = 4
    num_f_maps = 64
    kernel_size = 1
    dropout = 0.7
    
    # Training parameters
    batch_size = 16
    num_epochs = 100
    learning_rate = 5e-5
    weight_decay = 0.0001
    early_stopping_patience = 5
    validation_size = 0.2
    random_state = 42
    
    # Paths
    data_csv_path = '../../../ViT_balanced_train_cleaned.csv'
    save_dir = 'saved_models_2S4L16BS'
    analysis_dir = 'analysis_results_2S4L16BS'
    scaler_path = 'saved_models_2S4L16BS/scaler.save'  # path for scaler

    @staticmethod
    def get_config_dict():
        return {
            'features_dim': Config.features_dim,
            'num_classes': Config.num_classes,
            'num_stages': Config.num_stages,
            'num_layers': Config.num_layers,
            'num_f_maps': Config.num_f_maps,
            'kernel_size': Config.kernel_size,
            'dropout': Config.dropout,
            'batch_size': Config.batch_size,
            'num_epochs': Config.num_epochs,
            'learning_rate': Config.learning_rate,
            'weight_decay': Config.weight_decay,
            'validation_size': Config.validation_size,
            'random_state': Config.random_state
        }

def print_step(message, level=1):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "  " * (level-1) + "» " if level > 1 else ""
    print(f"[{timestamp}] {prefix}{message}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_step(f"Using device: {device}")

# Model Architecture (unchanged)
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, kernel_size, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.GELU(),
            nn.Conv1d(in_channels, out_channels, kernel_size, 
                     padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 1),
            nn.Dropout(dropout)
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AV_MSTCN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(config['features_dim'], config['num_f_maps'], 1),
            nn.BatchNorm1d(config['num_f_maps']),
            nn.GELU()
        )
        
        self.stages = nn.ModuleList([
            nn.Sequential(*[
                DilatedResidualLayer(2**i, config['num_f_maps'], config['num_f_maps'], 
                              config['kernel_size'], config['dropout'])
                for i in range(config['num_layers'])
            ]) for _ in range(config['num_stages'])
        ])
        
        self.attention = nn.Sequential(
            nn.Conv1d(config['num_f_maps'], config['num_f_maps']//4, 1),
            nn.GELU(),
            nn.Conv1d(config['num_f_maps']//4, 1, 1),
            nn.Softmax(dim=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(config['num_f_maps'], config['num_f_maps']//2),
            nn.LayerNorm(config['num_f_maps']//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config['num_f_maps']//2, config['num_classes'])
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x)
        
        attn_weights = self.attention(x)
        x = torch.sum(x * attn_weights, dim=2)
        return self.classifier(x)

class AV_Dataset(Dataset):
    def __init__(self, video_names, data_df, scaler=None, fit_scaler=False):
        self.video_names = video_names
        self.data_df = data_df.groupby('Video File')
        self.label_map = {'fake': 0, 'real': 1}
        self.scaler = scaler if scaler is not None else StandardScaler()
        self.fit_scaler = fit_scaler
        
        print_step(f"Initializing dataset with {len(video_names)} videos", level=2)
        
        if fit_scaler:
            print_step("Fitting scaler...", level=2)
            all_features = []
            for name in tqdm(video_names[:1000], desc="Collecting features"):
                features = self._get_features(name)
                all_features.append(features)
            self.scaler.fit(np.vstack(all_features))
    
    def _get_features(self, video_name):
        video_data = self.data_df.get_group(video_name)
        features = np.stack([ast.literal_eval(x) if isinstance(x, str) else x 
                       for x in video_data['Features']])
        return features
    
    def __len__(self):
        return len(self.video_names)
    
    def __getitem__(self, idx):
        video_name = self.video_names[idx]
        features = self._get_features(video_name)
            
        features = self.scaler.transform(features)
        label = self.label_map[self.data_df.get_group(video_name)['label'].iloc[0].lower().strip()]
        
        return torch.FloatTensor(features), label, video_name

def safe_collate(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    features, labels, video_names = zip(*batch)
    
    lengths = [f.shape[0] for f in features]
    max_len = max(lengths)
    padded_features = torch.zeros(len(batch), max_len, features[0].shape[1])
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded_features[i, :l] = f
    
    if random.random() > 0.3:
        padded_features += torch.randn_like(padded_features) * 0.01
        
    return padded_features, torch.LongTensor(labels), video_names, torch.tensor(lengths)

def validate(model, val_loader, criterion):
    model.eval()
    val_loss = 0
    all_labels = []
    all_probs = []
    all_preds = []
    
    with torch.no_grad():
        for features, labels, _, _ in val_loader:
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    val_loss /= len(val_loader)
    
    class_names = ['fake', 'real']
    report = classification_report(
        all_labels, 
        all_preds, 
        target_names=class_names,
        digits=4
    )
    
    return {
        'val_loss': val_loss,
        'val_acc': 100 * accuracy_score(all_labels, all_preds),
        'balanced_acc': 100 * balanced_accuracy_score(all_labels, all_preds),
        'val_auc': 100 * roc_auc_score(all_labels, np.array(all_probs)[:, 1]),
        'classification_report': report
    }

def train_model():
    os.makedirs(Config.save_dir, exist_ok=True)
    os.makedirs(Config.analysis_dir, exist_ok=True)
    
    # Load and prepare data
    df = pd.read_csv(Config.data_csv_path)
    video_info = df.groupby('Video File')['label'].first().reset_index()
    video_names = video_info['Video File'].values
    video_labels = video_info['label'].apply(lambda x: 0 if x.lower().strip() == 'fake' else 1).values
    
    train_videos, val_videos = train_test_split(
        video_names, 
        test_size=Config.validation_size,
        random_state=Config.random_state,
        stratify=video_labels
    )
    
    # Create and fit scaler on training data
    train_scaler = StandardScaler()
    train_dataset = AV_Dataset(train_videos, df, scaler=train_scaler, fit_scaler=True)
    
    # Save the scaler
    joblib.dump(train_scaler, Config.scaler_path)
    print_step(f"Scaler saved to {Config.scaler_path}", level=2)
    
    # Create validation dataset with the same scaler
    val_dataset = AV_Dataset(val_videos, df, scaler=train_scaler, fit_scaler=False)
    
    # Weighted sampling
    class_counts = np.bincount([train_dataset.label_map[df[df['Video File'] == name]['label'].iloc[0]] 
                              for name in train_videos])
    sample_weights = torch.tensor([1/class_counts[label] for label in video_labels[:len(train_videos)]])
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    # Data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.batch_size,
        sampler=sampler,
        collate_fn=safe_collate,
        pin_memory=False,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.batch_size,
        collate_fn=safe_collate,
        pin_memory=False,
        num_workers=0
    )
    
    # Initialize model and training
    model = AV_MSTCN(Config.get_config_dict()).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5)
    
    best_val_auc = 0
    history = []
    
    for epoch in range(Config.num_epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{Config.num_epochs}')
        for features, labels, _, _ in pbar:
            features = features.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'acc': f"{100.*correct/total:.2f}%"
            })
        
        # Validation
        val_metrics = validate(model, val_loader, criterion)
        scheduler.step(val_metrics['val_auc'])
        
        # Print results
        print_step(f"\nEpoch {epoch+1} Results:", level=2)
        print_step(f"Train Loss: {train_loss/len(train_loader):.4f} | Acc: {100.*correct/total:.2f}%", level=3)
        print_step(f"Val Loss: {val_metrics['val_loss']:.4f} | Acc: {val_metrics['val_acc']:.2f}%", level=3)
        print_step(f"Balanced Acc: {val_metrics['balanced_acc']:.2f}% | AUC: {val_metrics['val_auc']:.2f}%", level=3)
        print_step("\nClassification Report:\n" + val_metrics['classification_report'], level=3)
        
        history.append({
            'epoch': epoch+1,
            'train_loss': train_loss/len(train_loader),
            'train_acc': 100.*correct/total,
            **{k:v for k,v in val_metrics.items() if k != 'classification_report'}
        })
        
        # Save best model
        if val_metrics['val_auc'] > best_val_auc:
            best_val_auc = val_metrics['val_auc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_metrics': val_metrics,
                'config': Config.get_config_dict()
            }, os.path.join(Config.save_dir, 'best_model.pth'))
        
        # Early stopping
        if (epoch - np.argmax([h['val_auc'] for h in history])) > Config.early_stopping_patience:
            print_step(f"Early stopping at epoch {epoch+1}", level=2)
            break
    
    # Save history
    pd.DataFrame(history).to_csv(os.path.join(Config.analysis_dir, 'training_history.csv'), index=False)
    print_step(f"\nTraining completed. Best Val AUC: {best_val_auc:.2f}%")

if __name__ == '__main__':
    train_model()

[2025-07-03 21:56:01] Using device: cuda
[2025-07-03 21:56:24]   » Initializing dataset with 8063 videos
[2025-07-03 21:56:24]   » Fitting scaler...


[2025-07-03 21:57:24]   » Scaler saved to saved_models_2S4L16BS/scaler.save
[2025-07-03 21:57:24]   » Initializing dataset with 2016 videos


Epoch 1/100: 100%|██████████████████████████████████████████| 504/504 [07:41<00:00,  1.09it/s, loss=0.4137, acc=73.29%]


[2025-07-03 22:10:53]   » 
Epoch 1 Results:
[2025-07-03 22:10:53]     » Train Loss: 0.5593 | Acc: 73.29%
[2025-07-03 22:10:53]     » Val Loss: 0.4658 | Acc: 81.20%
[2025-07-03 22:10:53]     » Balanced Acc: 77.89% | AUC: 89.17%
[2025-07-03 22:10:53]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.7802    0.9525    0.8578      1200
        real     0.8966    0.6054    0.7228       816

    accuracy                         0.8120      2016
   macro avg     0.8384    0.7789    0.7903      2016
weighted avg     0.8273    0.8120    0.8031      2016



Epoch 2/100: 100%|██████████████████████████████████████████| 504/504 [08:08<00:00,  1.03it/s, loss=0.4396, acc=83.57%]


...

[2025-07-04 09:17:45]   » 
Epoch 71 Results:
[2025-07-04 09:17:45]     » Train Loss: 0.1464 | Acc: 98.85%
[2025-07-04 09:17:45]     » Val Loss: 0.1658 | Acc: 98.21%
[2025-07-04 09:17:45]     » Balanced Acc: 98.25% | AUC: 99.79%
[2025-07-04 09:17:45]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9891    0.9808    0.9849      1200
        real     0.9722    0.9841    0.9781       816

    accuracy                         0.9821      2016
   macro avg     0.9806    0.9825    0.9815      2016
weighted avg     0.9822    0.9821    0.9822      2016



Epoch 72/100: 100%|█████████████████████████████████████████| 504/504 [07:36<00:00,  1.10it/s, loss=0.1175, acc=98.62%]


[2025-07-04 09:27:13]   » 
Epoch 72 Results:
[2025-07-04 09:27:13]     » Train Loss: 0.1502 | Acc: 98.62%
[2025-07-04 09:27:13]     » Val Loss: 0.1580 | Acc: 98.51%
[2025-07-04 09:27:13]     » Balanced Acc: 98.36% | AUC: 99.77%
[2025-07-04 09:27:13]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9835    0.9917    0.9876      1200
        real     0.9876    0.9755    0.9815       816

    accuracy                         0.9851      2016
   macro avg     0.9855    0.9836    0.9845      2016
weighted avg     0.9851    0.9851    0.9851      2016



Epoch 73/100: 100%|█████████████████████████████████████████| 504/504 [07:36<00:00,  1.10it/s, loss=0.1174, acc=99.06%]


[2025-07-04 09:36:41]   » 
Epoch 73 Results:
[2025-07-04 09:36:41]     » Train Loss: 0.1394 | Acc: 99.06%
[2025-07-04 09:36:41]     » Val Loss: 0.1570 | Acc: 98.56%
[2025-07-04 09:36:41]     » Balanced Acc: 98.44% | AUC: 99.82%
[2025-07-04 09:36:41]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9851    0.9908    0.9880      1200
        real     0.9864    0.9779    0.9822       816

    accuracy                         0.9856      2016
   macro avg     0.9857    0.9844    0.9851      2016
weighted avg     0.9856    0.9856    0.9856      2016

[2025-07-04 09:36:41]   » Early stopping at epoch 73
[2025-07-04 09:36:41] 
Training completed. Best Val AUC: 99.84%


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd
from tqdm import tqdm
import ast
import random
import warnings
from datetime import datetime
from collections import Counter
import joblib  # For saving the scaler
import psutil

# Suppress warnings
warnings.filterwarnings('ignore')

class Config:
    # Data configuration
    features_dim = 256
    num_classes = 2
    
    # Model architecture
    num_stages = 2
    num_layers = 4
    num_f_maps = 64
    kernel_size = 1
    dropout = 0.7
    
    # Training parameters
    batch_size = 8
    num_epochs = 100
    learning_rate = 5e-5
    weight_decay = 0.0001
    early_stopping_patience = 5
    validation_size = 0.2
    random_state = 42
    
    # Paths
    data_csv_path = '../../../ViT_balanced_train_cleaned.csv'
    save_dir = 'saved_models_2S4L8BS'
    analysis_dir = 'analysis_results_2S4L8BS'
    scaler_path = 'saved_models_2S4L8BS/scaler.save'  # path for scaler

    @staticmethod
    def get_config_dict():
        return {
            'features_dim': Config.features_dim,
            'num_classes': Config.num_classes,
            'num_stages': Config.num_stages,
            'num_layers': Config.num_layers,
            'num_f_maps': Config.num_f_maps,
            'kernel_size': Config.kernel_size,
            'dropout': Config.dropout,
            'batch_size': Config.batch_size,
            'num_epochs': Config.num_epochs,
            'learning_rate': Config.learning_rate,
            'weight_decay': Config.weight_decay,
            'validation_size': Config.validation_size,
            'random_state': Config.random_state
        }

def print_step(message, level=1):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    prefix = "  " * (level-1) + "» " if level > 1 else ""
    print(f"[{timestamp}] {prefix}{message}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print_step(f"Using device: {device}")

# Model Architecture (unchanged)
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, in_channels, out_channels, kernel_size, dropout):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        
        self.conv = nn.Sequential(
            nn.BatchNorm1d(in_channels),
            nn.GELU(),
            nn.Conv1d(in_channels, out_channels, kernel_size, 
                     padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 1),
            nn.Dropout(dropout)
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
        
    def forward(self, x):
        return self.conv(x) + self.skip(x)

class AV_MSTCN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Conv1d(config['features_dim'], config['num_f_maps'], 1),
            nn.BatchNorm1d(config['num_f_maps']),
            nn.GELU()
        )
        
        self.stages = nn.ModuleList([
            nn.Sequential(*[
                DilatedResidualLayer(2**i, config['num_f_maps'], config['num_f_maps'], 
                              config['kernel_size'], config['dropout'])
                for i in range(config['num_layers'])
            ]) for _ in range(config['num_stages'])
        ])
        
        self.attention = nn.Sequential(
            nn.Conv1d(config['num_f_maps'], config['num_f_maps']//4, 1),
            nn.GELU(),
            nn.Conv1d(config['num_f_maps']//4, 1, 1),
            nn.Softmax(dim=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(config['num_f_maps'], config['num_f_maps']//2),
            nn.LayerNorm(config['num_f_maps']//2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(config['num_f_maps']//2, config['num_classes'])
        )
        
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.input_proj(x)
        
        for stage in self.stages:
            x = stage(x)
        
        attn_weights = self.attention(x)
        x = torch.sum(x * attn_weights, dim=2)
        return self.classifier(x)

class AV_Dataset(Dataset):
    def __init__(self, video_names, data_df, scaler=None, fit_scaler=False):
        self.video_names = video_names
        self.data_df = data_df.groupby('Video File')
        self.label_map = {'fake': 0, 'real': 1}
        self.scaler = scaler if scaler is not None else StandardScaler()
        self.fit_scaler = fit_scaler
        
        print_step(f"Initializing dataset with {len(video_names)} videos", level=2)
        
        if fit_scaler:
            print_step("Fitting scaler...", level=2)
            all_features = []
            for name in tqdm(video_names[:1000], desc="Collecting features"):
                features = self._get_features(name)
                all_features.append(features)
            self.scaler.fit(np.vstack(all_features))
    
    def _get_features(self, video_name):
        video_data = self.data_df.get_group(video_name)
        features = np.stack([ast.literal_eval(x) if isinstance(x, str) else x 
                       for x in video_data['Features']])
        return features
    
    def __len__(self):
        return len(self.video_names)
    
    def __getitem__(self, idx):
        video_name = self.video_names[idx]
        features = self._get_features(video_name)
            
        features = self.scaler.transform(features)
        label = self.label_map[self.data_df.get_group(video_name)['label'].iloc[0].lower().strip()]
        
        return torch.FloatTensor(features), label, video_name

def safe_collate(batch):
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    features, labels, video_names = zip(*batch)
    
    lengths = [f.shape[0] for f in features]
    max_len = max(lengths)
    padded_features = torch.zeros(len(batch), max_len, features[0].shape[1])
    for i, (f, l) in enumerate(zip(features, lengths)):
        padded_features[i, :l] = f
    
    if random.random() > 0.3:
        padded_features += torch.randn_like(padded_features) * 0.01
        
    return padded_features, torch.LongTensor(labels), video_names, torch.tensor(lengths)

def validate(model, val_loader, criterion):
    model.eval()
    val_loss = 0
    all_labels = []
    all_probs = []
    all_preds = []
    
    with torch.no_grad():
        for features, labels, _, _ in val_loader:
            features = features.to(device)
            labels = labels.to(device)
            
            outputs = model(features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    
    val_loss /= len(val_loader)
    
    class_names = ['fake', 'real']
    report = classification_report(
        all_labels, 
        all_preds, 
        target_names=class_names,
        digits=4
    )
    
    return {
        'val_loss': val_loss,
        'val_acc': 100 * accuracy_score(all_labels, all_preds),
        'balanced_acc': 100 * balanced_accuracy_score(all_labels, all_preds),
        'val_auc': 100 * roc_auc_score(all_labels, np.array(all_probs)[:, 1]),
        'classification_report': report
    }

def train_model():
    os.makedirs(Config.save_dir, exist_ok=True)
    os.makedirs(Config.analysis_dir, exist_ok=True)
    
    # Load and prepare data
    df = pd.read_csv(Config.data_csv_path)
    video_info = df.groupby('Video File')['label'].first().reset_index()
    video_names = video_info['Video File'].values
    video_labels = video_info['label'].apply(lambda x: 0 if x.lower().strip() == 'fake' else 1).values
    
    train_videos, val_videos = train_test_split(
        video_names, 
        test_size=Config.validation_size,
        random_state=Config.random_state,
        stratify=video_labels
    )
    
    # Create and fit scaler on training data
    train_scaler = StandardScaler()
    train_dataset = AV_Dataset(train_videos, df, scaler=train_scaler, fit_scaler=True)
    
    # Save the scaler
    joblib.dump(train_scaler, Config.scaler_path)
    print_step(f"Scaler saved to {Config.scaler_path}", level=2)
    
    # Create validation dataset with the same scaler
    val_dataset = AV_Dataset(val_videos, df, scaler=train_scaler, fit_scaler=False)
    
    # Weighted sampling
    class_counts = np.bincount([train_dataset.label_map[df[df['Video File'] == name]['label'].iloc[0]] 
                              for name in train_videos])
    sample_weights = torch.tensor([1/class_counts[label] for label in video_labels[:len(train_videos)]])
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
    
    # Data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=Config.batch_size,
        sampler=sampler,
        collate_fn=safe_collate,
        pin_memory=False,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=Config.batch_size,
        collate_fn=safe_collate,
        pin_memory=False,
        num_workers=0
    )
    
    # Initialize model and training
    model = AV_MSTCN(Config.get_config_dict()).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=Config.learning_rate, weight_decay=Config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5)
    
    best_val_auc = 0
    history = []
    
    for epoch in range(Config.num_epochs):
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{Config.num_epochs}')
        for features, labels, _, _ in pbar:
            features = features.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'acc': f"{100.*correct/total:.2f}%"
            })
        
        # Validation
        val_metrics = validate(model, val_loader, criterion)
        scheduler.step(val_metrics['val_auc'])
        
        # Print results
        print_step(f"\nEpoch {epoch+1} Results:", level=2)
        print_step(f"Train Loss: {train_loss/len(train_loader):.4f} | Acc: {100.*correct/total:.2f}%", level=3)
        print_step(f"Val Loss: {val_metrics['val_loss']:.4f} | Acc: {val_metrics['val_acc']:.2f}%", level=3)
        print_step(f"Balanced Acc: {val_metrics['balanced_acc']:.2f}% | AUC: {val_metrics['val_auc']:.2f}%", level=3)
        print_step("\nClassification Report:\n" + val_metrics['classification_report'], level=3)
        
        history.append({
            'epoch': epoch+1,
            'train_loss': train_loss/len(train_loader),
            'train_acc': 100.*correct/total,
            **{k:v for k,v in val_metrics.items() if k != 'classification_report'}
        })
        
        # Save best model
        if val_metrics['val_auc'] > best_val_auc:
            best_val_auc = val_metrics['val_auc']
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_metrics': val_metrics,
                'config': Config.get_config_dict()
            }, os.path.join(Config.save_dir, 'best_model.pth'))
        
        # Early stopping
        if (epoch - np.argmax([h['val_auc'] for h in history])) > Config.early_stopping_patience:
            print_step(f"Early stopping at epoch {epoch+1}", level=2)
            break
    
    # Save history
    pd.DataFrame(history).to_csv(os.path.join(Config.analysis_dir, 'training_history.csv'), index=False)
    print_step(f"\nTraining completed. Best Val AUC: {best_val_auc:.2f}%")

if __name__ == '__main__':
    train_model()

[2025-07-04 10:21:02] Using device: cuda
[2025-07-04 10:21:28]   » Initializing dataset with 8063 videos
[2025-07-04 10:21:28]   » Fitting scaler...


[2025-07-04 10:22:28]   » Scaler saved to saved_models_2S4L8BS/scaler.save
[2025-07-04 10:22:28]   » Initializing dataset with 2016 videos


Epoch 1/100: 100%|████████████████████████████████████████| 1008/1008 [09:35<00:00,  1.75it/s, loss=0.3780, acc=68.54%]


[2025-07-04 10:38:41]   » 
Epoch 1 Results:
[2025-07-04 10:38:41]     » Train Loss: 0.6033 | Acc: 68.54%
[2025-07-04 10:38:41]     » Val Loss: 0.4487 | Acc: 82.89%
[2025-07-04 10:38:41]     » Balanced Acc: 82.39% | AUC: 89.44%
[2025-07-04 10:38:41]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.8608    0.8500    0.8553      1200
        real     0.7834    0.7978    0.7905       816

    accuracy                         0.8289      2016
   macro avg     0.8221    0.8239    0.8229      2016
weighted avg     0.8294    0.8289    0.8291      2016



Epoch 2/100: 100%|████████████████████████████████████████| 1008/1008 [10:02<00:00,  1.67it/s, loss=0.7032, acc=80.48%]


...

[2025-07-04 18:01:42]   » 
Epoch 42 Results:
[2025-07-04 18:01:42]     » Train Loss: 0.1972 | Acc: 96.81%
[2025-07-04 18:01:42]     » Val Loss: 0.2029 | Acc: 96.73%
[2025-07-04 18:01:42]     » Balanced Acc: 97.11% | AUC: 99.71%
[2025-07-04 18:01:42]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9939    0.9508    0.9719      1200
        real     0.9320    0.9914    0.9608       816

    accuracy                         0.9673      2016
   macro avg     0.9630    0.9711    0.9663      2016
weighted avg     0.9689    0.9673    0.9674      2016



Epoch 43/100: 100%|███████████████████████████████████████| 1008/1008 [08:16<00:00,  2.03it/s, loss=0.1183, acc=97.22%]


[2025-07-04 18:12:00]   » 
Epoch 43 Results:
[2025-07-04 18:12:00]     » Train Loss: 0.1874 | Acc: 97.22%
[2025-07-04 18:12:00]     » Val Loss: 0.1813 | Acc: 97.52%
[2025-07-04 18:12:00]     » Balanced Acc: 97.72% | AUC: 99.76%
[2025-07-04 18:12:00]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9915    0.9667    0.9789      1200
        real     0.9527    0.9877    0.9699       816

    accuracy                         0.9752      2016
   macro avg     0.9721    0.9772    0.9744      2016
weighted avg     0.9758    0.9752    0.9753      2016



Epoch 44/100: 100%|███████████████████████████████████████| 1008/1008 [08:33<00:00,  1.96it/s, loss=0.6086, acc=96.76%]


[2025-07-04 18:22:40]   » 
Epoch 44 Results:
[2025-07-04 18:22:40]     » Train Loss: 0.1959 | Acc: 96.76%
[2025-07-04 18:22:40]     » Val Loss: 0.1916 | Acc: 96.97%
[2025-07-04 18:22:40]     » Balanced Acc: 97.28% | AUC: 99.77%
[2025-07-04 18:22:40]     » 
Classification Report:
              precision    recall  f1-score   support

        fake     0.9922    0.9567    0.9741      1200
        real     0.9395    0.9890    0.9636       816

    accuracy                         0.9697      2016
   macro avg     0.9658    0.9728    0.9689      2016
weighted avg     0.9709    0.9697    0.9699      2016

[2025-07-04 18:22:40]   » Early stopping at epoch 44
[2025-07-04 18:22:40] 
Training completed. Best Val AUC: 99.79%
